# 🎤 Voice Assistant Starter

**AI Learning Playground — Educational Quickstart Blueprint**

Build, run, and deploy a voice assistant
that combines Whisper speech-to-text with a local LLM for natural-language responses.

---

## What This Notebook Covers

| Step | Topic | Key Concept |
|------|-------|-------------|
| 1 | Environment Setup | Installing dependencies, verifying GPU + audio libs |
| 2 | Configure Settings | Loading `voice.yaml`, resolving model paths |
| 3 | Initialize Model | Instantiating `VoiceModel` with LlamaCpp + Whisper |
| 4 | Demo (text) | Text-mode: question directly to LLM |
| 5 | Demo (audio) | Audio-mode: WAV file → Whisper → LLM |
| 6 | GPU Monitoring | VRAM usage during inference |
| 7 | Register Model | Logging to MLflow as `AIStudio-EQ-Voice` |
| 8 | Verify | Loading registered model, text + audio round-trip |

## Voice Pipeline Architecture

```
┌─────────────────────────┐
│  Input (audio_base64)   │  ← base64-encoded WAV/MP3/OGG from Streamlit UI
└──────────┬──────────────┘
           │  if audio present
           ▼
   ┌───────────────┐
   │  Whisper STT  │  ← /home/jovyan/datafabric/whisper-large-v3
   └───────┬───────┘
           │  transcription text
           ▼
   ┌───────────────┐       ┌────────────────┐
   │  LLM (GGUF)  │  ←──  │  question text │  ← text fallback if no audio
   └───────┬───────┘       └────────────────┘
           │
           ▼
   ┌───────────────┐
   │ (answer, msgs)│
   └───────────────┘
```

In [ ]:
import sys
import time

sys.path.insert(0, "..")

start_time = time.time()
print("⏱️  Notebook started")

## 1. Environment Setup

In [ ]:
%pip install -q -r ../requirements.txt

import torch

cuda_available = torch.cuda.is_available()
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {'✅ Available' if cuda_available else '❌ Not found — CPU mode'}")

if cuda_available:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")

# Check audio processing library
try:
    import soundfile
    print("soundfile: ✅ Available")
except ImportError:
    print("soundfile: ❌ Not found — audio decoding may be limited")

## 2. Configure Settings

Load `configs/voice.yaml` with `capability: voice`.
Note the **two** model paths: `model_path` for the LLM and `stt_model_path` for Whisper.

In [ ]:
import os
from src.utils import load_config

config = load_config("../configs/voice.yaml")

model_path     = os.environ.get("MODEL_ARTIFACTS_PATH", config.get("model_path", ""))
stt_model_path = config.get("stt_model_path", "")
context_window = config.get("context_window", 8192)

print(f"Capability     : {config.get('capability')}")
print(f"LLM path       : {model_path}")
print(f"Whisper path   : {stt_model_path}")
print(f"Context window : {context_window} tokens")

## 3. Verify Assets

In [ ]:
from src.utils import log_asset_status

assets = [
    {"name": "LLM (GGUF)",     "path": model_path,                          "required": True},
    {"name": "Whisper model",  "path": stt_model_path,                      "required": False},
    {"name": "Config YAML",    "path": "../configs/voice.yaml",              "required": True},
    {"name": "Sample audio",   "path": "../data/input/sample_audio.wav",     "required": False},
    {"name": "Voice demo UI",  "path": "../demo/voice/main.py",              "required": False},
]

log_asset_status(assets)

## 4. Initialize VoiceModel

The `VoiceModel` loads the LLM at construction time.
Whisper is loaded lazily on the first audio request — saving VRAM until needed.

In [ ]:
from src.mlflow.models.voice import VoiceModel

print("Initializing VoiceModel...")

model = VoiceModel(
    config=config,
    model_path=model_path,
)

print("\n✅ VoiceModel ready")
print(f"   LLM loaded    : {'yes' if model.llm is not None else 'no (check model_path)'}")
print(f"   Whisper status: loads on first audio request")

## 5. Demo: Text Mode

When `audio_base64` is empty (or not provided), `VoiceModel` routes directly to the LLM
using the `question` field. This is the text fallback path.

In [ ]:
import pandas as pd

result = model.predict(pd.DataFrame([{
    "question":     "What is speech recognition and how does Whisper work?",
    "audio_base64": "",  # Empty → text mode
}]))

print("Q: What is speech recognition and how does Whisper work?")
print("\n" + "─" * 60)
print(result["answer"].iloc[0])

In [ ]:
# Multiple questions in text mode
questions = [
    "What is the difference between speech recognition and natural language processing?",
    "What types of audio formats does Whisper support?",
]

for q in questions:
    res = model.predict(pd.DataFrame([{"question": q, "audio_base64": ""}]))
    print(f"Q: {q}")
    print(f"A: {res['answer'].iloc[0][:300]}...")
    print()

## 6. Demo: Audio Mode

When `audio_base64` is provided, `VoiceModel`:
1. Decodes the base64 audio
2. Saves to a temporary WAV file
3. Transcribes with Whisper (`whisper-large-v3`)
4. Passes transcription + original question to the LLM

In [ ]:
import base64
import os

sample_audio_path = "../data/input/sample_audio.wav"

if os.path.exists(sample_audio_path):
    with open(sample_audio_path, "rb") as f:
        audio_b64 = base64.b64encode(f.read()).decode("utf-8")

    result = model.predict(pd.DataFrame([{
        "question":     "Please transcribe and summarize the audio content.",
        "audio_base64": audio_b64,
    }]))

    print("Audio mode result:")
    print("─" * 60)
    print(result["answer"].iloc[0])
else:
    print("ℹ️  No sample audio found at:", sample_audio_path)
    print("   Place any WAV file there to test audio mode.")
    print("   Text mode demonstrated above still exercises the full LLM pipeline.")

In [ ]:
import time
import plotly.graph_objects as go

# Benchmark text-mode response times across question lengths
questions_bench = [
    "Hello!",
    "What is speech recognition?",
    "Explain how automatic speech recognition works and what techniques are commonly used.",
]

lengths = [len(q.split()) for q in questions_bench]
times   = []

for q in questions_bench:
    t0 = time.time()
    model.predict(pd.DataFrame([{"question": q, "audio_base64": ""}]))
    times.append(time.time() - t0)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Response time (s)",
    x=[f"{w}w" for w in lengths],
    y=times,
    marker_color="#0096d6",
    text=[f"{t:.1f}s" for t in times],
    textposition="auto",
))
fig.update_layout(
    title="Voice Assistant Response Time (Text Mode)",
    xaxis_title="Question length (words)",
    yaxis_title="Response time (s)",
    template="plotly_dark",
)
fig.show()

## 7. GPU Monitoring

In [ ]:
from src.gpu_monitor import GPUMonitor

monitor = GPUMonitor()
monitor.display_dashboard()

## 8. Register with MLflow

Register as **`AIStudio-EQ-Voice`** — fully independent from the other three models.
The `capability: voice` key in the config routes serving requests to `VoiceModel`.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

ARTIFACT_PATH = "AIStudio-EQ-Voice"
MODEL_NAME    = "AIStudio-EQ-Voice"

print(f"Artifact path  : {ARTIFACT_PATH}")
print(f"Registered as  : {MODEL_NAME}")

In [ ]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# VoiceModel input schema: question text + optional base64 audio
input_schema = Schema([
    ColSpec("string", "question"),     # Text question (context or spoken intent)
    ColSpec("string", "audio_base64"), # Base64-encoded audio (empty = text-only mode)
])

output_schema = Schema([
    ColSpec("string", "answer"),    # LLM response
    ColSpec("string", "messages"),  # JSON conversation history
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print("Input  : question (string), audio_base64 (string)")
print("Output : answer (string), messages (JSON string)")

In [ ]:
from src.mlflow.logger import Logger

with mlflow.start_run(run_name=f"register-{ARTIFACT_PATH}") as run:
    Logger.log_model(
        signature     = signature,
        artifact_path = ARTIFACT_PATH,
        config_path   = "../configs/voice.yaml",
        model_path    = model_path,
        demo_folder   = "../demo/voice",
    )
    run_id = run.info.run_id

print(f"✅ Model logged | Run ID: {run_id}")

model_uri = f"runs:/{run_id}/{ARTIFACT_PATH}"
reg = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"✅ Registered  : {MODEL_NAME} v{reg.version}")

## 9. Verify Registration

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

test_result = loaded_model.predict(pd.DataFrame([{
    "question":     "What is the capital of France?",
    "audio_base64": "",
}]))

print("✅ Loaded model response:")
print(test_result["answer"].iloc[0][:300])

In [ ]:
elapsed = time.time() - start_time
print(f"⏱️  Total notebook time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

---

## ✅ What We Accomplished

| Step | Result |
|------|--------|
| Environment | CUDA + soundfile verified, dependencies installed |
| VoiceModel | Initialized with LlamaCpp (Whisper loads on demand) |
| Text demo | Multiple questions answered via LLM directly |
| Audio demo | WAV → Whisper → LLM pipeline demonstrated |
| GPU Monitor | VRAM usage during inference visible |
| Registration | `AIStudio-EQ-Voice` registered in Model Registry |
| Verification | Loaded model returned correct answer |

## All 4 Models Registered

| Model Name | Capability | Notebook |
|---|---|---|
| `AIStudio-EQ-Chatbot` | Conversational Q&A | `chatbot-starter.ipynb` |
| `AIStudio-EQ-ImageGen` | Text-to-image | `image-gen-starter.ipynb` |
| `AIStudio-EQ-Document` | Document RAG Q&A | `document-analyzer-starter.ipynb` |
| `AIStudio-EQ-Voice` | Voice assistant | `voice-assistant-starter.ipynb` |

## Next Steps

- **Deploy in AI Studio:** Open the Model Registry → select any registered model → Deploy → choose the matching Streamlit UI
- **Try real audio:** Record a WAV file and convert to base64 to test the Whisper pipeline
- **Extend the voice pipeline:** Add text-to-speech (TTS) for a full voice-in/voice-out experience